In [1]:
import pandapower as pp
import pp_heig_plot as pp_plot
import pp_heig_simulation as pp_sim
from pp_importer import PandaPowerImporter
import pandas as pd

net = pp.create.create_empty_network()
net_file_path = "list_batiments.xlsx"

In [2]:
# bus

batiments = pd.read_excel(net_file_path)

bus_transfo = pp.create_bus(net, vn_kv=0.4, name="bus_slack")

cabinet = batiments["Cabinet"].dropna().astype(str).str.strip()
cabinet = cabinet[cabinet.ne("")].unique()

for nom in cabinet:
    pp.create_bus(net, vn_kv=0.4, name=nom)

In [3]:
# charges

charge = batiments["NumÈro EGID"].dropna().astype(str).str.strip()
charge = charge[charge.ne("")].unique()

# Associe chaque nom de cabinet à l'index du bus correspondant dans net
bus_par_cabinet = (
    net.bus.dropna(subset=["name"]).reset_index().set_index("name")["index"].to_dict()
)

for _, ligne in batiments.iterrows():
    egid = ligne["NumÈro EGID"]
    cabinet = str(ligne["Cabinet"]).strip()

    if pd.isna(egid) or cabinet not in bus_par_cabinet:
        continue

    bus = bus_par_cabinet[cabinet]

    pp.create_load(
        net,
        bus=bus,
        p_mw=0.1,
        q_mvar=0.05,
        name=str(egid),
    )

In [7]:
# lignes

cables = [
    ("CDBT900784", "CDBT012139", 0.090, "GKT 3x50/50"),
    ("CDBT012139", "CDBT004760", 0.080, "GKT 3x50/50"),
    ("CDBT004760", "CDBT004764", 0.115, "GKN 3x95/95"),
    ("CDBT003746", "CDBT004764", 0.100, "GKN 3x150/150"),
    ("CDBT004764", "STMT003438", 0.180, "GKN 3x150/150"),
    ("STMT003438", "CDBT901452", 0.135, "GKN 3x150/150"),
    ("N1", "CDBT901452", 0.255, "GKN 3x50/50"),
    ("CDBT901452", "CDBT004774", 0.100, "GKN 3x95/95"),
    ("CDBT004774", "CDBT016055", 0.085, "GKN 3x50/50"),
    ("CDBT004774", "CDBT901604", 0.205, "GKN 3x150/150"),
    ("STMT003438", "60437", 0.340, "GKN 3x240/240"),
]

parametres_cables = {
    "GKT 3x50/50": (0.642, 0.083, 210, 0.142),
    "GKN 3x50/50": (0.641, 0.082, 205, 0.142),
    "GKN 3x95/95": (0.320, 0.080, 225, 0.220),
    "GKN 3x150/150": (0.206, 0.078, 245, 0.290),
    "GKN 3x240/240": (0.125, 0.075, 275, 0.415),
}


bus_par_nom = {str(nom).strip(): idx for idx, nom in net.bus["name"].items()}

for depart, arrivee, longueur_km, type_cable in cables:
    depart = str(depart).strip()
    arrivee = str(arrivee).strip()

    if depart not in bus_par_nom:
        raise ValueError(f"Bus de départ absent de net.bus : {depart}")

    if arrivee not in bus_par_nom:
        raise ValueError(f"Bus d'arrivée absent de net.bus : {arrivee}")

    if type_cable not in parametres_cables:
        raise ValueError(f"Paramètres inconnus pour le câble : {type_cable}")

    r, x, c, max_i = parametres_cables[type_cable]

    pp.create_line_from_parameters(
        net,
        from_bus=bus_par_nom[depart],
        to_bus=bus_par_nom[arrivee],
        length_km=longueur_km,
        r_ohm_per_km=r,
        x_ohm_per_km=x,
        c_nf_per_km=c,
        max_i_ka=max_i,
        name=f"{depart} - {arrivee} ({type_cable})",
    )

print(
    net.line[
        [
            "name",
            "from_bus",
            "to_bus",
            "length_km",
            "r_ohm_per_km",
            "x_ohm_per_km",
            "c_nf_per_km",
            "max_i_ka",
        ]
    ]
)

                                       name  from_bus  to_bus  length_km  \
0     CDBT900784 - CDBT012139 (GKT 3x50/50)        11       2      0.090   
1     CDBT012139 - CDBT004760 (GKT 3x50/50)         2       8      0.080   
2     CDBT004760 - CDBT004764 (GKN 3x95/95)         8       9      0.115   
3   CDBT003746 - CDBT004764 (GKN 3x150/150)        10       9      0.100   
4   CDBT004764 - STMT003438 (GKN 3x150/150)         9       5      0.180   
5   STMT003438 - CDBT901452 (GKN 3x150/150)         5       4      0.135   
6             N1 - CDBT901452 (GKN 3x50/50)         6       4      0.255   
7     CDBT901452 - CDBT004774 (GKN 3x95/95)         4       3      0.100   
8     CDBT004774 - CDBT016055 (GKN 3x50/50)         3       1      0.085   
9   CDBT004774 - CDBT901604 (GKN 3x150/150)         3       7      0.205   
10       STMT003438 - 60437 (GKN 3x240/240)         5      12      0.340   

    r_ohm_per_km  x_ohm_per_km  c_nf_per_km  max_i_ka  
0          0.642         0.083 

In [ ]:
# transfo de quartier

pp.create_transformer_from_parameters(
    net,
    hv_bus=,
    lv_bus=,
    sn_mva=0.4,
    vk_percent=6,
    vkr_percent=1.425,
    i0_percent=0.3375,
    pfe_kw=1.35,
    vn_hv_kv=20.0,
    vn_lv_kv=0.4,
    name="trafo1",
)

In [ ]:
# Noeud bilan

ext_grid = pp.create_ext_grid(net, bus=bus_transfo, vm_pu=1.02, name="slack")

In [10]:
from pandapower.plotting.plotly import simple_plotly

fig = simple_plotly(net, auto_open=False)
fig.show()

2026-10-19 11:13:14 macbook-pro-de-azteks-7.home pandapower.plotting.plotly.simple_plotly[14873] WARNING No or insufficient geodata available --> Creating artificial coordinates. This may take some time...


In [9]:
print("Bus :", len(net.bus))
print("Réseau externe :", len(net.ext_grid))
print("Lignes :", len(net.line))
print("Transformateurs :", len(net.trafo))
print(net.ext_grid)

Bus : 13
Réseau externe : 0
Lignes : 11
Transformateurs : 0
Empty DataFrame
Columns: [name, bus, vm_pu, va_degree, slack_weight, in_service, controllable]
Index: []
